<a href="https://colab.research.google.com/github/eliabrodsky/la_data/blob/main/Directed_Payment_Vulnerability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medicaid Directed Payments: Which Louisiana Rural Hospitals Are Most Exposed

Louisiana pays its rural hospitals roughly **$265 million a year** in Medicaid directed payments.
That pool is going to shrink.

H.R.1, signed July 2025, caps directed payments at 100% of the Medicare rate in expansion states.
Louisiana is an expansion state. Existing programs are grandfathered but reduced by 10 percentage
points a year starting with the first rating period after 1 January 2028. Louisiana's SFY2029 is
the first full year under the reduction. The same law also cuts the provider taxes that fund the
state share, so the pool shrinks from both ends.

This notebook identifies the hospitals that cannot absorb that, and sets out the three strategic
paths open to them with the evidence behind each.

**What it does not do.** It does not predict closure. It measures financial exposure and the
characteristics that make a hospital harder to sustain, and it names the hospitals where the
question is most urgent.

## Setup

In [ ]:
# ============ SETUP + HELPERS (tested standalone) ============
import numpy as np, pandas as pd, plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots

NAVY,TEAL,GOLD,RED,GREEN,BLUE,GREY = '#1F3864','#1C7293','#B08511','#E24B4A','#1D9E75','#378ADD','#888780'
OWNER = {'Government':NAVY,'Nonprofit':TEAL,'Proprietary':GOLD}
PROGRAM = {'Both':NAVY,'LIHNC only':TEAL,'Epic only':GOLD,'Neither':GREY}
RURAL = {'Metro / adjacent':NAVY,'Micropolitan':TEAL,'Small town / isolated':GOLD}

def style(fig, title, subtitle='', legend=None, height=520, left=90, bottom=None):
    """Consistent styling. Legend sits BELOW the plot so it can never overlap
    the chart area or the title. bottom margin grows to make room for it."""
    show_legend = legend is not False and any(
        tr.showlegend is not False and getattr(tr, 'name', None) for tr in fig.data)
    bottom = bottom if bottom is not None else (150 if show_legend else 80)
    fig.update_layout(
        title=dict(text=f'<b>{title}</b>' + (f'<br><sub>{subtitle}</sub>' if subtitle else ''),
                   x=0, xanchor='left', y=0.97, yanchor='top'),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(size=12, family='Aptos, Arial'),
        height=height, margin=dict(l=left, r=45, t=105, b=bottom),
        legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='left', x=0,
                    title='', bgcolor='rgba(0,0,0,0)') if show_legend else dict(),
        showlegend=show_legend)
    fig.update_xaxes(gridcolor='#e8e8e2', zeroline=False)
    fig.update_yaxes(gridcolor='#e8e8e2', zeroline=False)
    return fig

def quadrants(fig, x, y, labels, xr, yr):
    """Draw quadrant dividers and label each corner.
    labels order: bottom-left, bottom-right, top-left, top-right."""
    fig.add_vline(x=x, line=dict(color=GREY, width=1, dash='dash'))
    fig.add_hline(y=y, line=dict(color=GREY, width=1, dash='dash'))
    pos = [(xr[0], yr[0], 'left', 'bottom'), (xr[1], yr[0], 'right', 'bottom'),
           (xr[0], yr[1], 'left', 'top'),    (xr[1], yr[1], 'right', 'top')]
    for txt, (px_, py_, ax_, ay_) in zip(labels, pos):
        if not txt:
            continue
        fig.add_annotation(x=px_, y=py_, text=f'<b>{txt}</b>', showarrow=False,
                           xanchor=ax_, yanchor=ay_, font=dict(size=10, color=GREY),
                           bgcolor='rgba(255,255,255,0.75)')
    return fig

## Load the raw sources

In [ ]:
# ============ LOAD FROM RAW ============
BASE = 'https://raw.githubusercontent.com/eliabrodsky/la_data/main/'
FILES = {2021:'CostReport_2021_Final.csv',2022:'CostReport_2022_Final.csv',2023:'CostReport_2023_Final.csv'}
CONTROL = {1:'Nonprofit-Church',2:'Nonprofit-Other',3:'Proprietary-Individual',4:'Proprietary-Corp',
           5:'Proprietary-Partnership',6:'Proprietary-Other',7:'Gov-Federal',8:'Gov-City-County',
           9:'Gov-County',10:'Gov-State',11:'Gov-Hospital District',12:'Gov-City',13:'Gov-Other'}
PO_BOX = {'70511':'70510','70562':'70560','71121':'71220','70157':'70517','70308':'70380'}

def load_cost_report(year, path):
    d = pd.read_table(BASE+path, sep=',', header=0, low_memory=False)
    d = d[d['State Code']=='LA'].copy()
    col = lambda n: pd.to_numeric(d[n],errors='coerce') if n in d.columns else pd.Series(np.nan,index=d.index)
    begin=pd.to_datetime(d['Fiscal Year Begin Date'],errors='coerce')
    end=pd.to_datetime(d['Fiscal Year End Date'],errors='coerce')
    period=(end-begin).dt.days.clip(lower=1)
    cash=col('Cash on Hand and in Banks').fillna(0)+col('Temporary Investments').fillna(0)
    opex,dep=col('Less Total Operating Expense'),col('Depreciation Cost').fillna(0)
    npr,oth=col('Net Patient Revenue'),col('Total Other Income').fillna(0)
    total_days=col('Total Days (V + XVIII + XIX + Unknown)')
    o=pd.DataFrame({'fy':year,
        'ccn':d['Provider CCN'].astype(str).str.zfill(6),
        'hospital_filing_name':d['Hospital Name'].str.strip(),
        'city':d['City'].str.strip().str.title(),
        'medicare_class':d['CCN Facility Type'],
        'control':pd.to_numeric(d['Type of Control'],errors='coerce').map(CONTROL),
        'zip':d['Zip Code'].astype(str).str.extract(r'(\d{5})')[0].replace(PO_BOX),
        'reporting_days':period,'beds':col('Number of Beds'),'fte':col('FTE - Employees on Payroll'),
        'net_patient_revenue':npr,'operating_expense':opex,
        'medicaid_revenue':col('Net Revenue from Medicaid'),
        'uncompensated_care_cost':col('Cost of Uncompensated Care'),
        'cost_to_charge':col('Cost To Charge Ratio')})
    o['days_cash_on_hand']=(cash/((opex-dep)/period)).where((opex-dep)>0)
    o['equity_ratio']=(col('Total Fund Balances')/col('Total Assets')).where(col('Total Assets')>0)
    o['current_ratio']=(col('Total Current Assets')/col('Total Current Liabilities')).where(col('Total Current Liabilities')>0)
    o['patient_services_margin']=(col('Net Income from Service to Patients')/npr).where(npr>0)
    o['total_margin']=(col('Net Income')/(npr+oth)).where((npr+oth)>0)
    o['medicaid_days_share']=(col('Total Days Title XIX')/total_days).where(total_days>0)
    o['medicare_days_share']=(col('Total Days Title XVIII')/total_days).where(total_days>0)
    print(f'  FY{year}: {len(o)} Louisiana hospitals')
    return o

print('Loading CMS cost reports')
panel = pd.concat([load_cost_report(y,p) for y,p in FILES.items()], ignore_index=True)

xwalk = pd.read_table(BASE+'rural_ccn.csv', sep=',', header=0, dtype=str)
xwalk['ccn']=xwalk['ccn'].str.zfill(6)
panel = panel[panel.ccn.isin(set(xwalk.ccn))].copy()
panel['ownership'] = panel.control.map(lambda c:'Government' if str(c).startswith('Gov')
    else ('Proprietary' if str(c).startswith('Proprietary') else 'Nonprofit'))
print(f'{len(panel)} hospital-years across {panel.ccn.nunique()} rural hospitals')

pay = pd.read_table(BASE+'ldh_payments_2024_2025.csv',sep=',',header=0,low_memory=False)
pay['ccn']=pay.ccn.astype(str).str.zfill(6)
for y in (2024,2025):
    pay[f'ldh_payments_{y}']=pay[[f'dp_{y}',f'cs_{y}',f'upl_{y}',f'dsh_{y}']].fillna(0).sum(axis=1)

atlas = pd.read_table(BASE+'louisiana_health_atlas_export_zip.csv',sep=',',header=0,low_memory=False)
atlas['zip']=atlas['ZIP Code'].astype(str).str.zfill(5)
atlas=atlas.drop_duplicates('zip').rename(columns={
    'Population':'zip_population','Classification':'zip_classification',
    'Rural Designation (RUCA Category)':'ruca','Broadband Deserts':'broadband_desert_pct',
    'Social Vulnerability (Poverty)':'poverty_pct','Social Vulnerability (Food Access)':'food_access_pct',
    'Transportation (Vehicle Ownership)':'no_vehicle_pct',
    'Healthcare Facility (Specialty)':'specialty_facilities_zip',
    'Healthcare Facility (Acute)':'acute_facilities_zip','Diabetes Prevalence':'diabetes_prevalence'})
mco = pd.read_table(BASE+'parish_mco_enrollment.csv',sep=',',header=0)
print(f'{len(pay)} hospitals with LDH payments | {len(atlas)} ZIPs | {len(mco)} parishes')

## Build the hospital table

In [ ]:
# ============ BUILD THE ANALYSIS TABLE ============
latest = panel.sort_values('fy').groupby('ccn').tail(1).set_index('ccn')
wide = panel.pivot_table(index='ccn',columns='fy',
    values=['days_cash_on_hand','equity_ratio','current_ratio','patient_services_margin',
            'total_margin','net_patient_revenue'],aggfunc='first')
wide.columns=[f'{a}_fy{b}' for a,b in wide.columns]

XW = ['ccn','hospital','parish','lihnc_member','in_epic_pipeline','epic_wave','epic_status',
      'epic_est_pricing_m','epic_amb_vol','epic_ip_vol','epic_providers','medicaid_class']
XW = [c for c in XW if c in xwalk.columns]

h = xwalk[XW].set_index('ccn').join(wide).join(latest[[
    'city','zip','medicare_class','control','ownership','beds','fte','reporting_days',
    'net_patient_revenue','medicaid_revenue','uncompensated_care_cost','cost_to_charge',
    'medicaid_days_share','medicare_days_share']]).reset_index()

for c in ['epic_est_pricing_m','epic_amb_vol','epic_ip_vol','epic_providers']:
    h[c]=pd.to_numeric(h[c],errors='coerce')

h = h.merge(pay[['ccn','dp_2024','dp_2025','cs_2025','upl_2025','dsh_2025',
                 'ldh_payments_2024','ldh_payments_2025','adc_2024','discharges_2024']],
            on='ccn',how='left')
h = h.merge(atlas[['zip','zip_population','zip_classification','ruca','broadband_desert_pct',
                   'poverty_pct','food_access_pct','no_vehicle_pct','specialty_facilities_zip',
                   'acute_facilities_zip','diabetes_prevalence']],on='zip',how='left')
h['parish']=h.parish.astype(str).str.upper().str.replace('.','',regex=False)
h = h.merge(mco,on='parish',how='left')

# ---- normalization ----
h['medicare_class']=h.medicare_class.fillna('Unclassified')
h['program']=np.select(
    [(h.lihnc_member=='Yes')&(h.in_epic_pipeline=='Yes'),(h.lihnc_member=='Yes'),(h.in_epic_pipeline=='Yes')],
    ['Both','LIHNC only','Epic only'],default='Neither')
h['rurality_band']=pd.cut(h.ruca,[0,3,6,10],labels=['Metro / adjacent','Micropolitan','Small town / isolated'])

# size-normalized
h['revenue_per_bed']=h.net_patient_revenue_fy2023/h.beds.replace(0,np.nan)
h['revenue_per_fte']=h.net_patient_revenue_fy2023/h.fte.replace(0,np.nan)
h['fte_per_bed']=h.fte/h.beds.replace(0,np.nan)
h['occupancy_rate']=h.adc_2024/h.beds.replace(0,np.nan)

# share-of-revenue measures
h['medicaid_share_of_revenue']=h.medicaid_revenue/h.net_patient_revenue_fy2023
h['uncompensated_share_of_revenue']=h.uncompensated_care_cost/h.net_patient_revenue_fy2023
h['state_payment_share_of_revenue']=h.ldh_payments_2025/h.net_patient_revenue_fy2023
h['epic_cost_share_of_revenue']=h.epic_est_pricing_m*1e6/h.net_patient_revenue_fy2023
h['epic_cost_per_provider']=h.epic_est_pricing_m*1e6/h.epic_providers

# derived
h['non_care_margin_gap']=h.total_margin_fy2023-h.patient_services_margin_fy2023
h['days_cash_change_21_23']=h.days_cash_on_hand_fy2023-h.days_cash_on_hand_fy2021
h['medicaid_revenue_at_risk']=h.medicaid_revenue*h.mco_pct_change_24_26.abs()
h['social_vulnerability_index']=h[['poverty_pct','food_access_pct','no_vehicle_pct','broadband_desert_pct']].mean(axis=1)

# within-ownership percentile ranks
for c in ['days_cash_on_hand_fy2023','equity_ratio_fy2023','current_ratio_fy2023']:
    h[c+'_rank']=h.groupby('ownership')[c].rank(pct=True)

h=h.copy()
print(f'analysis table: {h.shape[0]} hospitals, {h.shape[1]} columns')
print(h.program.value_counts().to_string())

---
# 1 &nbsp; How large is the exposure

Before naming hospitals, size the thing. Four payment streams reach these hospitals from LDH.
One of them is almost the whole amount.

In [ ]:
# WHAT THE STATE PAYS, AND THROUGH WHICH MECHANISM
streams = pd.DataFrame({
    'stream': ['Directed payments', 'DSH', 'UPL', 'Cost settlement'],
    'sfy2025': [h.dp_2025.sum(), h.dsh_2025.sum(), h.upl_2025.sum(), h.cs_2025.sum()]})
streams['share'] = streams.sfy2025 / streams.sfy2025.sum()
streams['sfy2025'] = streams.sfy2025.round(0)
print(streams.to_string(index=False))
print()
print(f'Hospitals receiving directed payments: {int((h.dp_2025.fillna(0) > 0).sum())}')
print(f'Median share of net patient revenue:    {h.state_payment_share_of_revenue.median():.1%}')
print(f'Highest share:                          {h.state_payment_share_of_revenue.max():.1%}')

**Directed payments are 95% of it.** Cost settlement, the mechanism usually credited with keeping
cost-based rural hospitals solvent, is under $1 million statewide. Whatever protection a Critical
Access Hospital gets from cost-based reimbursement does not extend to this.

In [ ]:
# WHAT A REDUCTION DOES TO THE BOTTOM LINE
h['net_income_fy2023'] = h.total_margin_fy2023 * h.net_patient_revenue_fy2023
already_negative = int((h.net_income_fy2023 < 0).sum())

rows = []
for cut in (0.10, 0.20, 0.30, 0.50, 1.00):
    removed = h.dp_2025.fillna(0) * cut
    after = h.net_income_fy2023 - removed
    rows.append({'reduction': f'{cut:.0%}',
                 'removed_$M': round(removed.sum() / 1e6, 1),
                 'hospitals_negative': int((after < 0).sum()),
                 'newly_negative': int((after < 0).sum()) - already_negative})

print(f'Already reporting a negative bottom line in FY2023: {already_negative} '
      f'of {int(h.net_income_fy2023.notna().sum())}')
print()
print(pd.DataFrame(rows).to_string(index=False))

**The damage is not linear.** A 10% reduction pushes only three more hospitals into the red,
because most of the affected hospitals have some cushion. Past 30% the cushion is gone and the
count climbs steeply. The phase-down reaches 30% in its third year.

---
# 2 &nbsp; Who cannot absorb it

Exposure is not just dependence. A hospital that leans heavily on the payment but holds a year of
cash has time to restructure. One that leans heavily and holds three weeks does not.

Six characteristics, each ranked across the 64 hospitals so that 1.0 is the most vulnerable
position and 0.0 the least:

| Component | Why it matters |
|---|---|
| Payment dependence | Share of revenue from state Medicaid payments |
| Liquidity | Days cash on hand. How long it can operate while adjusting |
| Margin | Whether it is already losing money |
| Trend | Direction of liquidity over three years |
| Market size | Population of the hospital ZIP. Smaller means fewer alternatives locally, and a thinner base to rebuild on |
| Scale | Net patient revenue. Small hospitals have less fixed cost to cut before they cut capability |

The index is the mean of the six. It is a screening tool, not a prediction.

In [ ]:
# VULNERABILITY INDEX
def rank_pct(series, invert=False):
    """Percentile rank, 0 to 1. invert=True makes LOW values score HIGH."""
    r = series.rank(pct=True)
    return (1 - r) if invert else r

h['v_dependence'] = rank_pct(h.state_payment_share_of_revenue)
h['v_liquidity']  = rank_pct(h.days_cash_on_hand_fy2023, invert=True)
h['v_margin']     = rank_pct(h.total_margin_fy2023, invert=True)
h['v_trend']      = rank_pct(h.days_cash_change_21_23, invert=True)
h['v_market']     = rank_pct(h.zip_population, invert=True)
h['v_scale']      = rank_pct(h.net_patient_revenue_fy2023, invert=True)

COMPONENTS = ['v_dependence', 'v_liquidity', 'v_margin', 'v_trend', 'v_market', 'v_scale']
h['vulnerability'] = h[COMPONENTS].mean(axis=1)

h['tier'] = pd.cut(h.vulnerability, [0, .45, .60, 1.0],
                   labels=['Lower exposure', 'Watch', 'Highest exposure'])

print(h.tier.value_counts().reindex(['Highest exposure', 'Watch', 'Lower exposure']).to_string())
print()
print(h.groupby('tier', observed=True)[
    ['state_payment_share_of_revenue', 'days_cash_on_hand_fy2023',
     'total_margin_fy2023', 'zip_population', 'ruca']].median().round(3).to_string())

In [ ]:
# THE NAMED LIST
cols = ['hospital', 'parish', 'medicare_class', 'state_payment_share_of_revenue',
        'days_cash_on_hand_fy2023', 'total_margin_fy2023', 'zip_population',
        'ruca', 'adc_2024', 'program', 'vulnerability']

highest = h[h.tier == 'Highest exposure'].sort_values('vulnerability', ascending=False)
print(f'{len(highest)} hospitals in the highest exposure tier')
print(highest[cols].round(3).to_string(index=False))

**Six of these hospitals are in neither LIHNC nor the Epic pipeline.** The programs the state is
running are not reaching the hospitals with the most to lose, which is consistent with the earlier
finding that program membership is unrelated to financial position.

In [ ]:
# WHERE EACH HOSPITAL SITS: DEPENDENCE AGAINST TIME TO REACT
x = h.dropna(subset=['state_payment_share_of_revenue', 'days_cash_on_hand_fy2023']).copy()
x['ldh_payments_2025'] = x.ldh_payments_2025.fillna(0)

fig = px.scatter(x, x='state_payment_share_of_revenue', y='days_cash_on_hand_fy2023',
                 color='tier', size='ldh_payments_2025', hover_name='hospital',
                 text=np.where(x.tier == 'Highest exposure', x.hospital, ''),
                 color_discrete_map={'Highest exposure': RED, 'Watch': GOLD,
                                     'Lower exposure': TEAL}, size_max=30)
fig.update_traces(textposition='middle right', textfont=dict(size=8))
fig.update_xaxes(title='State Medicaid payments as a share of net patient revenue', tickformat='.0%')
fig.update_yaxes(title='Days cash on hand (FY2023)', range=[-60, 420])
quadrants(fig, 0.15, 90,
          ['Low dependence,<br>little time to react', 'High dependence, little time to react',
           'Low dependence, has time', 'High dependence, has time'],
          [0, x.state_payment_share_of_revenue.max()], [-60, 420])
fig = style(fig, 'Directed payment dependence against time to react',
            'Bubble size is total SFY2025 state payments. Dashed lines at 15% of revenue and 90 days cash',
            height=620)
fig.show()

---
# 3 &nbsp; Path one: join a broader network

## What the evidence says

Affiliation is the most studied of the three paths, and the findings cut both ways.

**It reduces closure risk.** AHA analysis of UNC Sheps Center closure data finds that closures have
disproportionately affected independent hospitals, and that rural hospitals are less likely to
close after joining a system than if they remain independent.

**It can improve quality.** A case-control study in JAMA Network Open comparing merged rural
hospitals to independent ones found decreases in inpatient mortality for heart failure, acute
myocardial infarction, stroke and pneumonia.

**It costs service lines.** The strongest counter-finding. Analysis of 172 rural hospitals that
merged between 2009 and 2016 across 32 states (Health Affairs, 2021) found merged hospitals were
more likely than unaffiliated ones to eliminate maternal, neonatal and surgical care within one to
two years. Mental health and substance use volumes fell or held steady at merged hospitals while
rising at unaffiliated ones, suggesting unmet need in those communities.

**Financial benefit is not guaranteed.** RUPRI's 2024 review of the affiliation landscape and work
summarized by Equitable Growth both note that evidence on post-merger profitability, capital
structure and debt is limited and mixed.

**A network is not a merger.** LIHNC is a membership network, not an acquisition. It offers shared
contracting, shared analytics and shared infrastructure without transferring control or triggering
the service line consolidation that the merger literature describes. That distinction matters when
presenting this option to a parish board.

## Who this fits

In [ ]:
# HOSPITALS THAT WOULD GAIN MOST FROM NETWORK MEMBERSHIP
# Criteria: meaningful exposure, still has some operating base, and not already in a network
candidates = h[(h.tier.isin(['Highest exposure', 'Watch'])) &
               (h.program.isin(['Neither', 'Epic only'])) &
               (h.adc_2024.fillna(0) >= 1)]

print(f'{len(candidates)} exposed hospitals with real inpatient volume and no network membership')
print(candidates[['hospital', 'parish', 'state_payment_share_of_revenue',
                  'days_cash_on_hand_fy2023', 'adc_2024', 'ruca', 'program']]
      .sort_values('state_payment_share_of_revenue', ascending=False).round(3).to_string(index=False))

---
# 4 &nbsp; Path two: restructure operations

## What the evidence says

Restructuring covers a wide range, from staffing adjustments to changing the licence itself.

**Rural Emergency Hospital conversion** is the formal version. A hospital gives up inpatient beds,
keeps a 24/7 emergency department and outpatient services, and receives a fixed monthly facility
payment from Medicare plus enhanced outpatient rates. It was created by Congress in 2021 and took
effect in 2023, with technical assistance available through the REH Technical Assistance Center.

For several Louisiana hospitals this is closer to a formalization than a change.

**Freestanding emergency departments are not a substitute.** NC Rural Health Research Program work
found the freestanding ED model generally not viable in rural areas due to low volumes, high
uninsured rates and provider shortages. MedPAC found nearly all stand-alone EDs are in urban and
more affluent areas. REH exists precisely because that gap needed a Medicare payment mechanism.

**Service reduction has downstream effects.** The closure literature repeatedly finds that losing
obstetric services corresponds to increases in preterm births, and that reducing inpatient
psychiatric beds is associated with higher suicide rates, though causality is not established.
Restructuring that removes a service line is not a neutral financial decision.

## Who this fits

In [ ]:
# HOSPITALS ALREADY OPERATING AS OUTPATIENT FACILITIES
# Average daily census under 1 means the inpatient service has effectively stopped
hollow = h[(h.adc_2024.notna()) & (h.adc_2024 < 1)].sort_values('adc_2024')

print(f'{len(hollow)} hospitals average under one inpatient per day')
print(hollow[['hospital', 'parish', 'medicare_class', 'discharges_2024', 'adc_2024',
              'beds', 'dp_2025', 'days_cash_on_hand_fy2023', 'ruca']].round(2).to_string(index=False))
print()
print('Staffing relative to the census actually being run')
staffing = h.dropna(subset=['fte_per_bed', 'occupancy_rate']).copy()
staffing['fte_per_occupied_bed'] = staffing.fte / staffing.adc_2024.replace(0, np.nan)
print(staffing.nlargest(10, 'fte_per_occupied_bed')[
    ['hospital', 'beds', 'adc_2024', 'fte', 'fte_per_occupied_bed', 'occupancy_rate']]
    .round(2).to_string(index=False))

**Read this carefully before acting on it.** A hospital with high FTE per occupied bed is not
necessarily overstaffed. A 24/7 emergency department in an isolated parish requires a minimum crew
whether or not anyone is admitted, and that standby cost is the point of the facility. What the
figures identify is hospitals whose cost structure is built around inpatient care they no longer
deliver, which is the situation REH conversion was designed for.

The distinction to test locally: is the staffing supporting emergency and outpatient demand, or
maintaining an inpatient capability that is no longer used?

---
# 5 &nbsp; Path three: what happens if neither works

## What the evidence says

The literature on rural hospital closure is uneven but consistent on a few points.

**Travel distance increases, reliably.** A 2025 Health Affairs Scholar study using Medicare claims
from 2010 to 2020 found median travel distance for common surgical conditions rose from 13.1 to
16.4 miles for beneficiaries who lost their nearest rural hospital. Notably, it found *no*
significant change in 30-day mortality, complications or readmissions for those surgical
conditions.

**Emergency response slows.** Miller and colleagues (Health Services Research, 2020) found closures
increased EMS response and transport times, with one study reporting emergency responder activation
time rising by 7.2 minutes.

**Mortality findings are mixed and condition-specific.** Gujral and Basu (NBER, 2019) found
increases in inpatient mortality following California rural closures. Carroll found increased
mortality for time-sensitive conditions. Other work finds no population-level effect, and some
suggests longer travel can be offset by arriving at a higher-quality hospital. The systematic
review by Mills and colleagues (Journal of Rural Health, 2024) concludes the comprehensive impact
is not well studied and that outcome measures are inconsistent across studies.

**Economic effects are clearer than clinical ones.** Closures are associated with reduced local
labour force, reduced population, and increased unemployment. Physician supply falls when a rural
hospital closes.

**Second-order effects reach other providers.** Work published in 2025 found rural closures affect
nursing home residents through longer transport for both emergent and urgent admissions.

## Sizing it for Louisiana

In [ ]:
# WHAT A SERVICE AREA LOSES
# Population served is approximated by the ZIP population, which understates the true
# catchment. Isolation is the RUCA category: 10 is the most isolated classification.
at_risk = h[h.tier == 'Highest exposure'].copy()

print(f'{len(at_risk)} highest-exposure hospitals')
print(f'  combined ZIP population:        {at_risk.zip_population.sum():,.0f}')
print(f'  combined FTE employees:         {at_risk.fte.sum():,.0f}')
print(f'  combined net patient revenue:   ${at_risk.net_patient_revenue_fy2023.sum()/1e6:,.0f}M')
print(f'  combined state payments:        ${at_risk.ldh_payments_2025.sum()/1e6:,.0f}M')
print(f'  combined annual discharges:     {at_risk.discharges_2024.sum():,.0f}')
print()
print('Most isolated of the exposed group (RUCA 10 is small town, most isolated)')
print(at_risk[at_risk.ruca >= 7][
    ['hospital', 'parish', 'ruca', 'zip_population', 'acute_facilities_zip',
     'fte', 'discharges_2024']].sort_values('ruca', ascending=False).round(0).to_string(index=False))

**The gap in this measure, stated plainly.** Isolation here is RUCA plus the count of acute
facilities in the same ZIP. Neither tells you how far the next emergency department actually is.
Geocoding the cost report addresses and computing a drive-time matrix would replace this proxy with
the number that matters, and it is the single most useful addition to this analysis.

Until then, treat RUCA 10 with a low ZIP population as a flag for further examination rather than
as a measured access risk.

---
# 6 &nbsp; Matching hospitals to paths

No hospital has only one option, and the three paths are not mutually exclusive. The screen below
is a starting point for a conversation, not an assignment.

In [ ]:
# SUGGESTED STARTING POINT BY PROFILE
def suggest(r):
    if pd.notna(r.adc_2024) and r.adc_2024 < 1:
        return 'Review REH conversion'
    if r.program in ('Neither', 'Epic only') and pd.notna(r.adc_2024) and r.adc_2024 >= 1:
        return 'Network membership first'
    if r.program in ('Both', 'LIHNC only'):
        return 'In a network; focus on operations'
    return 'Insufficient data'

exposed = h[h.tier.isin(['Highest exposure', 'Watch'])].copy()
exposed['starting_point'] = exposed.apply(suggest, axis=1)

print(exposed.starting_point.value_counts().to_string())
print()
for path in exposed.starting_point.unique():
    grp = exposed[exposed.starting_point == path].sort_values('vulnerability', ascending=False)
    print(f'--- {path}  ({len(grp)}) ---')
    print(grp[['hospital', 'parish', 'state_payment_share_of_revenue',
               'days_cash_on_hand_fy2023', 'adc_2024', 'ruca', 'program']]
          .round(2).to_string(index=False))
    print()

---
# What this analysis cannot tell you

1. **Drive times.** Isolation is proxied by RUCA and facilities in the same ZIP. The real question
   is how far the next emergency department is, which requires geocoding and a drive-time matrix.
2. **Referral flows.** LDH inpatient discharge data would show where these hospitals already send
   patients, which is the natural map for network formation.
3. **Local commitments.** Parish tax millage, outstanding bond obligations and physician contracts
   all constrain what a board can actually do, and none appear in a cost report.
4. **Closure prediction.** This measures exposure. Many financially distressed rural hospitals
   operate for years without closing, and the literature is explicit that poor financial performance
   alone is a weak predictor.
5. **Data vintage.** Cost reports are FY2023, payments SFY2025, enrollment through May 2026.

## Sources

Mills, Kaiser et al., *The impact of rural general hospital closures on communities: a systematic
review*, Journal of Rural Health, 2024 &middot; Health Affairs Scholar, *Changes in surgical quality
and access after rural hospital closures*, 2025 &middot; Miller, James, Holmes et al., *The effect
of rural hospital closures on emergency medical service response and transport times*, Health
Services Research, 2020 &middot; Gujral and Basu, *Impact of Rural and Urban Hospital Closures on
Inpatient Mortality*, NBER, 2019 &middot; Health Affairs, *Access to obstetric, behavioral health
and surgical inpatient services after hospital mergers in rural areas*, 2021 &middot; RUPRI,
*Health System Affiliation Landscape*, 2024 &middot; *Service changes, utilization and financial
performance after critical access hospitals join hospital systems*, 2025 &middot; NC Rural Health
Research Program and MedPAC on freestanding emergency department viability &middot; Rural Health
Redesign Center and Mathematica, REH conversion technical assistance.